# مرحلهٔ دوم — میز DBA

پنج پرسش Olist را با پلن واقعی و زمان‌بندی تکرارشونده بررسی می‌کنیم. مسیر نهایی پرسش درآمد از Snapshot است؛ چهار پرسش دیگر زنده محاسبه می‌شوند. منطق آزمایش در ماژول‌های پروژه قرار دارد.

**هشدار: این آزمایش schemaهای `olist` و `analytics` را جایگزین می‌کند. `DATABASE_URL` باید به یک دیتابیس آزمایشی مستقل اشاره کند، نه دیتابیس مرحلهٔ سوم یا محیط عملیاتی.** اجرای دوباره، مبنای بدون ایندکس آزمایشی را از نو می‌سازد؛ هیچ مرحلهٔ دستی در pgAdmin لازم نیست. فایل‌های خام Olist و خروجی مرحلهٔ اول پیش‌نیاز هستند.

## ۰. تنظیمات و مرز آزمایش
رمز فقط از متغیر محیطی خوانده می‌شود؛ اتصال پیش‌فرض یا رمز در کد نداریم. seed ثابت است. زمان بارگذاری اولیه جدا از سقف ده‌دقیقه‌ای محاسبه می‌شود. پنج نمونه بعد از یک گرم‌کردن گرفته می‌شود؛ هم میانه و هم بیشینه گزارش می‌شوند، چون میانهٔ خوب تضمین همهٔ درخواست‌ها نیست.

In [1]:
import json
import os
import random
import sys
import time
from datetime import date
from pathlib import Path
from sqlalchemy import create_engine, text

SEED = 42
random.seed(SEED)
notebook_started = time.perf_counter()
cwd = Path.cwd()
STAGE_ROOT = cwd if cwd.name == 'stage-2-database-performance' else cwd / 'part2' / 'stage-2-database-performance'
PROJECT_ROOT = STAGE_ROOT.parents[1]
sys.path.insert(0, str(STAGE_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if not os.environ.get('DATABASE_URL'):
    raise RuntimeError('Set DATABASE_URL to a disposable PostgreSQL database before starting Jupyter.')
from database_performance.experiments import prepare_database, collect_experiment, bridge_demo, show_query_evidence, save_experiment
engine = create_engine(os.environ['DATABASE_URL'], pool_pre_ping=True)
START_DATE, END_DATE = date(2016, 9, 1), date(2018, 11, 1)
REPETITIONS = 5
print('Seed:', SEED, '| repetitions:', REPETITIONS)
with engine.connect() as connection:
    print(connection.execute(text('SELECT version()')).scalar_one())

Seed: 42 | repetitions: 5


PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## ۱. بارگذاری تراکنشی و زنجیرهٔ مرحلهٔ اول
در دیتابیس آزمایشی، فقط schemaهای مرحلهٔ دوم و جدول ثبت migration آن بازسازی می‌شوند. دادهٔ مرحلهٔ اول از طریق پکیج خودش در schema جدا بارگذاری می‌شود؛ نبود فایل ورودی خطاست و پنهان نمی‌شود. تعداد ردیف خام، پذیرفته و قرنطینه‌شده شواهد تصمیم یکپارچگی هستند. این سلول بارگذاری اولیه است و زمانش از بودجهٔ اجرا کنار گذاشته می‌شود.

In [2]:
load_result = prepare_database(engine, STAGE_ROOT / 'data' / 'raw', PROJECT_ROOT / 'data' / 'curated' / 'curated_records.csv')
print(json.dumps(load_result, ensure_ascii=False, indent=2))

{
  "base_migrations": [
    "001_olist.sql",
    "002_analytics.sql"
  ],
  "olist": {
    "datasets": {
      "category_translations": {
        "source": 71,
        "loaded": 71,
        "quarantined": 0
      },
      "customers": {
        "source": 99441,
        "loaded": 99441,
        "quarantined": 0
      },
      "sellers": {
        "source": 3095,
        "loaded": 3095,
        "quarantined": 0
      },
      "products": {
        "source": 32951,
        "loaded": 32938,
        "quarantined": 13
      },
      "orders": {
        "source": 99441,
        "loaded": 99441,
        "quarantined": 0
      },
      "order_items": {
        "source": 112650,
        "loaded": 112626,
        "quarantined": 24
      },
      "order_payments": {
        "source": 103886,
        "loaded": 103886,
        "quarantined": 0
      },
      "order_reviews": {
        "source": 99224,
        "loaded": 99224,
        "quarantined": 0
      },
      "geolocation": {
        "source"

## ۲. آزمایش کنترل‌شده: مبنا، ایندکس، بازنویسی، بازنشستگی
در مبنا، Q1 از View زنده و Q5 از نسخهٔ اولیه استفاده می‌کنند؛ Q4 از ابتدا نسخهٔ مجموعه‌ای دارد تا آزمایش به هزاران اسکن همبسته تبدیل نشود. سپس همان SQL با ایندکس‌ها اندازه‌گیری می‌شود، بازنویسی پرداخت جدا سنجیده می‌شود، ایندکس‌های بی‌استفاده با بررسی نامشان در پلن حذف می‌شوند و مسیر نهایی Snapshot اجرا می‌شود.
برای دادهٔ تاریخی کم‌تغییر، `VACUUM (ANALYZE)` پیش از اندازه‌گیری، آمار و visibility map را آماده می‌کند؛ هزینهٔ این نگهداری در زمان کل آزمایش هست. این فرض در دادهٔ پرتغییر باید دوباره ارزیابی شود. طرح هر مرحله با `EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)` ذخیره می‌شود. مقدار کامل نتیجه نیز در هر نمونه دریافت می‌شود.

In [3]:
experiment = collect_experiment(engine, START_DATE, END_DATE, repetitions=REPETITIONS)
experiment['seed'] = SEED
experiment['load'] = load_result
print(json.dumps({'environment': experiment['environment'], 'query_comparison': experiment['query_comparison'], 'equivalence': experiment['equivalence']}, ensure_ascii=False, indent=2))

{
  "environment": {
    "postgresql_version": "PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit",
    "work_mem": "4MB",
    "shared_buffers": "128MB",
    "parallel_workers_per_gather": "2"
  },
  "query_comparison": {
    "01_monthly_revenue": {
      "baseline_ms": 360.405,
      "indexed_ms": 358.625,
      "rewritten_live_ms": 385.533,
      "final_ms": 4.345,
      "final_max_ms": 6.482,
      "median_under_200ms": true,
      "all_samples_under_200ms": true
    },
    "02_late_delivery": {
      "baseline_ms": 125.368,
      "indexed_ms": 124.622,
      "rewritten_live_ms": 126.589,
      "final_ms": 130.165,
      "final_max_ms": 141.324,
      "median_under_200ms": true,
      "all_samples_under_200ms": true
    },
    "03_delay_reviews": {
      "baseline_ms": 165.724,
      "indexed_ms": 131.507,
      "rewritten_live_ms": 125.953,
      "final_ms": 127.73,
      "final_max_ms": 137.447,
      "median_under

## ۳. پرسش درآمد: وقتی Index کافی نیست
فیلتر بازهٔ تقریباً کامل، تجمیع ماه و دسته و `count(DISTINCT)` را حذف نمی‌کند. پلن زنده قبل و بعد از ایندکس را با پلن نهایی Snapshot مقایسه کنید: اسکن و تجمیع چندجدولی به خواندن نتیجهٔ ماهانه تبدیل می‌شود. هزینهٔ انتخاب، تازگی غیرلحظه‌ای، فضای اضافه و بازسازی است. برابری همهٔ ردیف‌ها با View در همین بازه بررسی می‌شود؛ برای تغییر داده باید Procedure دوباره اجرا شود.

In [4]:
show_query_evidence(experiment, '01_monthly_revenue')

baseline
{
  "sql": "SELECT month, category, revenue, order_count, average_order_value\nFROM analytics.monthly_category_revenue\nWHERE month >= :start_date AND month < :end_date\nORDER BY month, category;\n",
  "row_count": 1264,
  "client_elapsed_ms": [
    359.405,
    350.297,
    362.764,
    377.05,
    360.405
  ],
  "client_median_ms": 360.405,
  "client_max_ms": 377.05,
  "server_plan": {
    "Node Type": "Aggregate",
    "Strategy": "Sorted",
    "Partial Mode": "Simple",
    "Parallel Aware": false,
    "Async Capable": false,
    "Startup Cost": 6053.44,
    "Total Cost": 6133.56,
    "Plan Rows": 547,
    "Plan Width": 108,
    "Actual Startup Time": 369.658,
    "Actual Total Time": 414.611,
    "Actual Rows": 1264,
    "Actual Loops": 1,
    "Group Key": [
      "((date_trunc('month'::text, o.order_purchase_timestamp))::date)",
      "(COALESCE(t.product_category_name_english, p.product_category_name, '__uncategorized__'::text))"
    ],
    "Shared Hit Blocks": 718118,
  

## ۴. پرسش تحویل: ایالت و ماه
مخرج فقط سفارش تحویل‌شده با هر دو تاریخ معتبر است؛ تأخیر بر حسب روز تقویمی تعریف می‌شود. این کوئری از View استفاده می‌کند. بازهٔ بزرگ ممکن است Hash Join و Seq Scan را توجیه کند؛ صرف وجود ایندکس به معنی بهترشدن آن نیست. تخمین ردیف و مقدار واقعی و تفاوت زمان قبل و بعد، مبنای قضاوت هستند.

In [5]:
show_query_evidence(experiment, '02_late_delivery')

baseline
{
  "sql": "-- ============================================================================\n-- Query 02: Late Delivery Rate by Customer State\n-- ============================================================================\n-- Business Context:\n--   Monitors logistics performance and SLA compliance across Brazilian states.\n--   Computes monthly delivered order volumes, late delivery counts, and late ratio.\n--\n-- Performance & Execution Characteristics:\n--   - Underlying View: analytics.order_delivery_reviews\n--   - Partial Index Support: `idx_orders_delivered_purchase` on olist.orders\n--     (order_purchase_timestamp WHERE order_status = 'delivered') allows rapid\n--     index range scan filtering directly to the date interval.\n--   - Join Optimization: Joins orders with customers on customer_id to extract\n--     customer_state. Supported by `idx_orders_customer`.\n--   - Modern SQL Syntax: Uses `FILTER (WHERE ...)` rather than cumbersome CASE\n--     statements for 

## ۵. پرسش رضایت: یک وزن برای هر سفارش
نظرهای متعدد ابتدا در سطح سفارش میانگین گرفته می‌شوند؛ سپس میانگین سفارش‌ها در گروه تأخیر محاسبه می‌شود. اتصال غیرضروری مشتری وجود ندارد. ایندکس پوششی نظرها زمانی مفید است که در پلن `Index Only Scan` و Heap Fetches کم دیده شود؛ مقایسهٔ ردیف‌های تخمینی و واقعی را هم نگه می‌داریم.

In [6]:
show_query_evidence(experiment, '03_delay_reviews')

baseline
{
  "sql": "-- ============================================================================\n-- Query 03: Customer Review Score Distribution Across Delivery Delay Buckets\n-- ============================================================================\n-- Business Context:\n--   Quantifies the impact of shipping delays on customer satisfaction (NPS / CSAT).\n--   Segments delivered orders into 3 delay tiers and calculates mean review score.\n--\n-- Performance & Execution Characteristics:\n--   - Fan-out Prevention: Reviews are aggregated per `order_id` in an inline subquery\n--     BEFORE joining with orders. This ensures orders with multiple reviews do not\n--     falsify order counts or artificially multiply order weights.\n--   - Immutable Helper: Uses `analytics.delivery_delay_days(actual, promised)`,\n--     which is pure SQL, IMMUTABLE, and PARALLEL SAFE, enabling parallel query plans.\n--   - Covered Indexing: Benefited by `idx_order_reviews_order` which includes\n--  

## ۶. پرسش فروشندگان: محدودسازی قبل از غنی‌سازی
ابتدا بیست فروشندهٔ برتر با درآمد تعیین می‌شوند. سپس سفارش‌های یکتای همان فروشندگان و میانگین نظر هر سفارش محاسبه می‌شوند؛ سفارش چندقلمی وزن اضافی نمی‌گیرد. CTEهای Materialized جلوی انتشار تخمین کم‌دقت را می‌گیرند و نظرها مجموعه‌ای خوانده می‌شوند، نه با اسکن همبسته برای هر سفارش. در این آزمایش، مبنای جاری هم همین بازنویسی امن است؛ زمان نسخهٔ تاریخی LATERAL به‌عنوان اندازه‌گیری جدید ادعا نمی‌شود.

In [7]:
show_query_evidence(experiment, '04_top_sellers')

baseline
{
  "sql": "-- Query 04: Top 20 sellers by delivered merchandise revenue in [start_date, end_date).\n-- Keep one row per seller/order for preparation time and review weighting.\n-- Materialization prevents the small top-20 estimate from propagating through\n-- the distinct-order/review joins. Reviews are aggregated once, not once per order.\nWITH top_sellers AS MATERIALIZED (\n    SELECT\n        i.seller_id,\n        sum(i.price) AS revenue\n    FROM olist.order_items AS i\n    JOIN olist.orders AS o USING (order_id)\n    WHERE o.order_status = 'delivered'\n      AND o.order_purchase_timestamp >= :start_date\n      AND o.order_purchase_timestamp < :end_date\n    GROUP BY i.seller_id\n    ORDER BY revenue DESC, i.seller_id\n    LIMIT 20\n), seller_orders AS MATERIALIZED (\n    SELECT DISTINCT\n        i.seller_id,\n        i.order_id,\n        o.order_approved_at,\n        o.order_delivered_carrier_date\n    FROM olist.order_items AS i\n    JOIN top_sellers USING (seller_id)\n

## ۷. پرسش پرداخت: جلوگیری از تکثیر اقلام و پرداخت‌ها
سطح تجمیع `(order_id, payment_type, payment_installments)` حفظ می‌شود؛ چند پرداخت هم‌نوع یک سفارش، چند وزن برای ارزش سبد نمی‌سازد. اقلام جدا جمع می‌شوند و سبد بدون قلم NULL می‌ماند. سه ایندکس پوششی در پلن نهایی باید اسکن مرتب و بدون مراجعهٔ زیاد به heap بدهند؛ `ORDER BY` میانی جریان اتصال را نگه می‌دارد. برابری دقیق نسخهٔ اولیه و بازنویسی بررسی می‌شود. فضای ایندکس و هزینهٔ نوشتن، بهای این انتخاب است.

In [8]:
show_query_evidence(experiment, '05_payments')

baseline
{
  "sql": "-- ============================================================================\n-- Query 05: Payment Methods, Installment Distribution, and Basket Value\n-- ============================================================================\n-- Business Context:\n--   Analyzes payment instrument usage patterns (credit card, boleto, voucher,\n--   debit card), installment adoption, total transaction counts, and compares\n--   actual payment values against total order merchandise value (items + freight).\n--\n-- Performance & Execution Characteristics:\n--   - Two-Level Hierarchical Aggregation:\n--     1. Inner subquery `grouped`: Groups payments per `(order_id, payment_type,\n--        payment_installments)` to collapse multi-voucher or multi-card transactions\n--        at the order level without causing Cartesian fan-out against order items.\n--     2. Pre-aggregated Order Value: Subquery `v` aggregates item prices + freight\n--        per `order_id` in isolation befor

## ۸. هزینهٔ پنهان Index و مسیر ردشده
حجم تک‌تک ایندکس‌ها، مجموع و نسبت به جدول در سه مرحله ذخیره می‌شود. candidateهای استفاده‌نشده در پلن‌های بازهٔ کامل کنار گذاشته می‌شوند؛ این تصمیم تضمین سودمندی برای بازه‌های باریک‌تر نیست. باقی‌ماندن View درآمد بالای بودجه بعد از ایندکس، مسیر ناموفق ثبت‌شدهٔ ماست.
آزمایش نوشتن، درج همان پنجاه‌هزار ردیف پرداخت در جدول موقت با کلید اصلی را با حالت دارای ایندکس پوششی مقایسه می‌کند؛ سه بار و با جابه‌جایی ترتیب. ساخت جدول و ایندکس و parsing در زمان درج نیست. این عدد اثر ایندکس پرداخت را می‌سنجد، نه هزینهٔ همهٔ ایندکس‌ها روی تمام جدول‌ها.

In [9]:
print(json.dumps({key: experiment[key] for key in ['indexes_seen_before_retirement', 'retired_indexes', 'storage_before', 'storage_candidates', 'storage_final', 'write_cost', 'snapshot_refresh_ms']}, ensure_ascii=False, indent=2))

{
  "indexes_seen_before_retirement": [
    "idx_order_items_order_value_cover",
    "idx_order_items_seller",
    "idx_order_payments_order_method_cover",
    "idx_order_reviews_order",
    "idx_orders_order_purchase_cover",
    "order_items_pkey",
    "products_pkey",
    "sellers_pkey"
  ],
  "retired_indexes": [
    "idx_geolocation_zip",
    "idx_order_items_product",
    "idx_order_payments_order",
    "idx_orders_customer",
    "idx_orders_delivered_purchase",
    "idx_products_category"
  ],
  "storage_before": {
    "tables": [
      {
        "table_name": "category_translations",
        "table_bytes": 16384,
        "index_bytes": 16384,
        "total_bytes": 32768
      },
      {
        "table_name": "customers",
        "table_bytes": 8470528,
        "index_bytes": 4333568,
        "total_bytes": 12804096
      },
      {
        "table_name": "geolocation",
        "table_bytes": 87080960,
        "index_bytes": 22487040,
        "total_bytes": 109568000
      },
   

## ۹. پل Python و SQL، Function و Procedure
خواندن View با cursor جریانی و chunk انجام می‌شود؛ خلاصهٔ ایالت از همان دادهٔ واقعی استخراج و با نگاشت صریح نوع‌ها نوشته می‌شود. Function در SELECT اختلاف روز را برمی‌گرداند؛ Procedure با CALL اثر نوشتن دارد و دوبار برای یک ماه اجرا و مقایسه می‌شود. تابع این پروژه خالص است، اما این ویژگی همهٔ Functionهای PostgreSQL نیست: تابع VOLATILE هم می‌تواند داده بنویسد؛ کنترل تراکنش در تابع مجاز نیست. Procedure ما داخل تراکنش فراخواننده اجرا می‌شود، نه با COMMIT مستقل. تست جداگانه، شکست میانهٔ درج و rollback را بررسی می‌کند.

In [10]:
experiment['bridge'] = bridge_demo(engine, START_DATE, END_DATE)
print(json.dumps(experiment['bridge'], ensure_ascii=False, indent=2))

{
  "chunk_sizes": [
    20000,
    20000,
    20000,
    20000,
    16470
  ],
  "total_view_rows": 96470,
  "state_summary_rows": 27,
  "function_delay_days": 2,
  "procedure_idempotent": true,
  "snapshot_month_rows": 43
}


## ۱۰. نتیجه و محدودیت ادعا
میانه و بیشینهٔ همهٔ پرسش‌ها جدا گزارش می‌شوند. نمونهٔ بالای بودجه پنهان یا با عدد اجرای دیگری جایگزین نمی‌شود. زمان کل بدون بارگذاری اولیه باید کمتر از ده دقیقه باشد. اتصال‌ها context-managed هستند و Engine در پایان بسته می‌شود. فایل JSON شاهد تکمیلی است؛ همهٔ اعداد لازم در خروجی ذخیره‌شدهٔ همین نوت‌بوک هم حضور دارند.

In [11]:
experiment['runtime_seconds_excluding_initial_load'] = round(time.perf_counter() - notebook_started - load_result['initial_load_seconds'], 3)
experiment['runtime_under_600_seconds'] = experiment['runtime_seconds_excluding_initial_load'] < 600
try:
    save_experiment(experiment, STAGE_ROOT / 'data' / 'curated' / 'performance_results.json')
    print(json.dumps({'query_comparison': experiment['query_comparison'], 'runtime_seconds_excluding_initial_load': experiment['runtime_seconds_excluding_initial_load'], 'runtime_under_600_seconds': experiment['runtime_under_600_seconds']}, indent=2))
    print('Saved data/curated/performance_results.json')
finally:
    engine.dispose()

{
  "query_comparison": {
    "01_monthly_revenue": {
      "baseline_ms": 360.405,
      "indexed_ms": 358.625,
      "rewritten_live_ms": 385.533,
      "final_ms": 4.345,
      "final_max_ms": 6.482,
      "median_under_200ms": true,
      "all_samples_under_200ms": true
    },
    "02_late_delivery": {
      "baseline_ms": 125.368,
      "indexed_ms": 124.622,
      "rewritten_live_ms": 126.589,
      "final_ms": 130.165,
      "final_max_ms": 141.324,
      "median_under_200ms": true,
      "all_samples_under_200ms": true
    },
    "03_delay_reviews": {
      "baseline_ms": 165.724,
      "indexed_ms": 131.507,
      "rewritten_live_ms": 125.953,
      "final_ms": 127.73,
      "final_max_ms": 137.447,
      "median_under_200ms": true,
      "all_samples_under_200ms": true
    },
    "04_top_sellers": {
      "baseline_ms": 148.181,
      "indexed_ms": 165.232,
      "rewritten_live_ms": 157.272,
      "final_ms": 155.485,
      "final_max_ms": 173.662,
      "median_under_200ms"